In [5]:
from typing import List, Optional
from pydantic import BaseModel, Field
from grounded_ai import EvaluationError, EvaluationInput, Evaluator

# --- 1. Define Custom Evaluation Schema ---
# The power of the library is defining exactly HOW you want to judge the output.
# Here we define a strict schema for checking Hallucinations/Groundedness.


class Citation(BaseModel):
    statement: str = Field(description="The specific statement made in the response")
    is_supported: bool = Field(
        description="True if the statement is supported by the context"
    )
    evidence: Optional[str] = Field(
        description="Quote from context supporting the statement, if applicable"
    )


class GroundednessEvaluation(BaseModel):
    score: float = Field(
        description="numerical value between min(0.0) to max(1.0) score where 1.0 is fully grounded in context"
    )
    reasoning: str = Field(description="High level explanation of the score")
    citations: List[Citation] = Field(
        description="Line-by-line analysis of the response"
    )

In [6]:
# 2. Initialize Evaluator with an AWS Bedrock model
    # Requires AWS credentials (AWS_PROFILE or env vars)
evaluator = Evaluator(
    model="bedrock/openai.gpt-oss-safeguard-120b",
    region_name="us-east-1",
)

In [7]:
# 3. Setup the RAG Scenario
# We want to check if the 'response' is actually supported by the 'context'.

rag_context = """
    Product: SuperBattery 3000
    Specs:
    - Battery Life: 24 hours on standard load.
    - Charging: Fast charge 0-80% in 15 minutes.
    - Weight: 150 grams.
    - Warranty: 2 years limited warranty.
    - Colors: Black, Silver, and Midnight Blue.
"""

user_query = "Tell me about the battery life and warranty of the SuperBattery."

# A response that contains a hallucination ("Red" color is not in context, "5 year" warranty is wrong)
ai_response = "The SuperBattery 3000 lasts for 24 hours. It comes with a 5-year warranty and is available in Red."

In [8]:
# 4. Run Evaluation
# We pass the input as an EvaluationInput object which formats the prompt automatically using Jinja2
input_payload = EvaluationInput(
    query=user_query, context=rag_context, response=ai_response
)

# The backend will force the LLM to output exactly matching our GroundednessEvaluation schema
result = evaluator.evaluate(
    input_data=input_payload,
    output_schema=GroundednessEvaluation,
    system_prompt="You are a strict groundedness evaluator. Verify every claim against the context.",
)

if isinstance(result, EvaluationError):
    print(f"\nEvaluation failed with error: {result.error_code}")
    print(f"Message: {result.message}")


# 5. Review Results
print("\n--- Evaluation Result ---")
print(f"Score: {result.score}/1.0")
print(f"Reasoning: {result.reasoning}\n")

print("Citations Analysis:")
for citation in result.citations:
    status = "✅" if citation.is_supported else "❌"
    print(f' {status} Statement: "{citation.statement}"')
    if citation.is_supported:
        print(f'    Evidence: "{citation.evidence}"')



--- Evaluation Result ---
Score: 0.45/1.0
Reasoning: The response addresses the query but contains factual errors and adds irrelevant misinformation.

Citations Analysis:
 ✅ Statement: "The SuperBattery 3000 lasts for 24 hours."
    Evidence: "Battery Life: 24 hours on standard load."
 ❌ Statement: "It comes with a 5‑year warranty"
 ❌ Statement: "is available in Red"
